In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class AgentState(TypedDict):
    num1: int
    num2: int
    num3: int
    result1: int
    result2: int
    op1: str
    op2: str

def addition1(state: AgentState) -> AgentState:
    """This is the addition node of the graph"""
    state['result1'] = state['num1'] + state['num2']
    return state

def addition2(state: AgentState) -> AgentState:
    """This is the addition node of the graph"""
    state['result2'] = state['num3'] + state['result1']
    return state

def subtraction1(state: AgentState) -> AgentState:
    """This is the subtraction node of the graph"""
    state['result1'] = state['num1'] - state['num2']
    return state

def subtraction2(state: AgentState) -> AgentState:
    """This is the subtraction node of the graph"""
    state['result2'] = state['num3'] - state['result1']
    return state

def router1(state: AgentState) -> str:
    """This is the router node of the graph"""
    if state['op1'] == '+':
        return "addition_e1"
    elif state['op1'] == '-':
        return "subtraction_e1"

def router2(state: AgentState) -> str:
    """This is the router node of the graph"""
    if state['op2'] == '+':
        return "addition_e2"
    elif state['op2'] == '-':
        return "subtraction_e2"

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("addition_n1", addition1)
graph.add_node("addition_n2", addition2)
graph.add_node("subtraction_n1", subtraction1)
graph.add_node("subtraction_n2", subtraction2)

graph.add_node("routing1", lambda state: state)
graph.add_node("routing2", lambda state: state)

graph.add_edge(START, "routing1")
graph.add_edge("addition_n1", "routing2")
graph.add_edge("subtraction_n1", "routing2")
graph.add_edge("addition_n2", END)
graph.add_edge("subtraction_n2", END)

graph.add_conditional_edges(
    "routing1",
    router1,
    {
        "addition_e1": "addition_n1",
        "subtraction_e1": "subtraction_n1",
    },
)

graph.add_conditional_edges(
    "routing2",
    router2,
    {
        "addition_e2": "addition_n2",
        "subtraction_e2": "subtraction_n2",
    },
)

app = graph.compile()

In [ ]:
result = app.invoke({"num1": 1, "num2": 2, "num3": 3, "op1": "+", "op2": "+"})

result['result2']